In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

✅ Project root: c:\Users\123\Desktop\start_vector
✅ Sys path: ['c:\\Users\\123\\Desktop\\start_vector', 'C:\\Python314\\python314.zip', 'C:\\Python314\\DLLs']...


In [ ]:
from src_oop.core.my_gspread import GoogleTabs
from src_oop.jobs.annual_procurement_plan.config import delivery_calculation_china, annual_procurement_plan
from src_oop.core.utils_general import clean_currency_value

import pandas as pd
from datetime import datetime
import gspread
import numpy as np

In [38]:
class Annual_procurement_plan:
    def __init__(self):
        # Получаем данные из Google Sheets Заказы белые ТЕСТ
        self.table_from = delivery_calculation_china.get("title")
        self.sheet_from = delivery_calculation_china.get("white_orders_sheet")
        self.google_connect_from = GoogleTabs(self.table_from, self.sheet_from)
        # Получаем данные из Google Sheets Заказы
        self.table_from_orders = delivery_calculation_china.get("title")
        self.sheet_from_orders = delivery_calculation_china.get("orders_sheet")
        self.google_connect_from_orders = GoogleTabs(self.table_from_orders, self.sheet_from_orders)
        # Данные для вставки в таблицу Годовой план закупа 2026
        self.table_to = annual_procurement_plan.get("title")
        self.sheet_to = annual_procurement_plan.get("orders_sheet")
        self.google_connect_to = GoogleTabs(self.table_to, self.sheet_to)
        # Колонки для фильтрации
        self.choosen_orders_columns = ["wild", "Модель", "Статус", "Кол-во к заказу", "Сумма заказа, RMB", "нед прибытие"]

    def get_white_orders_data(self):
        # Подключаемся к листу Заказы белые ТЕСТ и получаем данные
        data = self.google_connect_from.sheet_title.get_all_values()
        data_from = data[4:] # Данные начиная с 5-й строки
        columns_from = data[3] # Названия колонок с 4-й строки
        df = pd.DataFrame(data_from, columns=columns_from)
        return df
    
    def get_orders_data(self):
        # Подключаемся к листу Заказы и получаем данные
        data = self.google_connect_from_orders.sheet_title.get_all_values()
        data_from = data[7:] # Данные начиная с 8-й строки
        columns_from = data[6] # Названия колонок с 7-й строки
        df = pd.DataFrame(data_from, columns=columns_from)
        return df

    def set_data(self, df):
        # Вставляем данные в таблицу Годовой план закупа 2026 в лист БД Заказы
        self.google_connect_to.set_df_to_google(df)

In [39]:
# Создаем экземпляр класса и получаем данные
plan = Annual_procurement_plan()
df_white_orders = plan.get_white_orders_data()
# Выбираем нужные колонки
choosen_orders_columns = plan.choosen_orders_columns
df_white_orders_short = df_white_orders[choosen_orders_columns]

✅ Успешное подключение к Расчет поставки Китай_по обороту -> Заказы белые ТЕСТ
✅ Успешное подключение к Расчет поставки Китай_по обороту -> Заказы
✅ Успешное подключение к Годовой план закупа 2026 -> БД_ЗАКАЗЫ


In [ ]:
# Получаем датафрейм от листа Заказы в таблице Расчет поставки Китай_по обороту
df_orders = plan.get_orders_data()
df_orders_short = df_orders[choosen_orders_columns]

In [41]:
# Объединяем датафреймы вертикально
df_merge = pd.concat([
    df_white_orders_short.reset_index(drop=True), 
    df_orders_short.reset_index(drop=True)
], ignore_index=True)
# Убираем знаки валюты из колонки
df_merge['Сумма заказа, RMB'] = df_merge['Сумма заказа, RMB'].apply(clean_currency_value)
# Выбираем статусы для фильтрации
cancel_statuses = ["отмена", "в планах", "прибыло"]
# Фильтрация
df_merge = df_merge.loc[~df_merge['Статус'].isin(cancel_statuses)]
# Добавляем колонку, указывающую на время обновления
df_merge["updatet_at"] = (datetime.now()).strftime("%Y-%m-%d %H:%M:%S")

,wild,Модель,Статус,Кол-во к заказу,"Сумма заказа, RMB",нед прибытие,updatet_at
0,wild1630,Помпа BM-WP02 белая (400mah),производство окончено,2 560,18432.0,16,2026-04-13 15:08:33
1,wild1631,Помпа BM-WP02 белая (800mah),производство окончено,2 560,21504.0,16,2026-04-13 15:08:33
12,wild1634,Помпа белая BM-WP10 (600mah),производство окончено,5 040,68040.0,23,2026-04-13 15:08:33
13,wild1635,Помпа белая BM-WP10 (800mah),производство окончено,10 080,156240.0,23,2026-04-13 15:08:33
14,wild1371,Водонагреватель- кран из нержавеющей стали BM-...,производство окончено,5 004,310248.0,23,2026-04-13 15:08:33


In [42]:
# Обновляем данные в таблице Годовой план закупа 2026
plan.set_data(df_merge)

✅ Успешное подключение к Годовой план закупа 2026 -> БД_ЗАКАЗЫ
Данные вставлены в гугл таблицу
